In [1]:
# =============================================================================
# 单元格 1: 启动模块 (环境初始化、全局配置与并行 Worker)
# =============================================================================

import os
import re
import glob
import json
import torch
import warnings
import pandas as pd
import numpy as np
import multiprocessing
from pebble import ProcessPool
from concurrent.futures import TimeoutError
from concurrent.futures import ProcessPoolExecutor, as_completed
from sklearn.decomposition import PCA
from tqdm.auto import tqdm 
from transformers import AutoTokenizer, AutoModel


# --- RDKit ---
from rdkit import Chem, RDLogger
import utils

# --- 1. 全局环境与静音设置 ---
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings("ignore", category=RuntimeWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.set_num_threads(1)  # 防止 PyTorch 底层多线程与多进程打架

# --- 2. 全局超参数与路径配置 ---
MAX_HEAVY_ATOMS_FOR_SMALL_MOL = 200 
TARGET_EMBEDDING_DIM = 768 

# 自动探测系统核心数，为防止几十万原子的巨型分子导致 OOM，上限锁定在 64 或更高
DETECTED_CORES = multiprocessing.cpu_count()
NUM_WORKERS = min(DETECTED_CORES, 64) 

OUTPUT_VOCAB_768 = os.path.join('processed_data', 'fragment_vocabulary_768.json')
OUTPUT_VOCAB_PCA = os.path.join('processed_data', 'fragment_vocabulary_128_pca.json')
os.makedirs(os.path.dirname(OUTPUT_VOCAB_768), exist_ok=True)

HF_MODEL_NAME = "seyonec/ChemBERTa-Zinc-Base-v1"
LOCAL_MODEL_PATH = os.path.join(os.getcwd(), "ChemBERTa_Local")

# --- 3. 核心辅助函数 ---
def fast_get_shards_smiles(fragment):
    smiles_list = []
    try:
        pre_calc_data = utils.prepare_molecule_for_cutting(fragment)
        if not pre_calc_data: return smiles_list 
        
        main_mol = pre_calc_data.get('main_mol_for_matching')
        ion_fragments = pre_calc_data.get('ion_fragments', [])

        for ion in ion_fragments:
            try:
                rw_ion = Chem.RWMol(ion)
                for atom in rw_ion.GetAtoms():
                    if atom.GetAtomicNum() == 0: atom.SetAtomicNum(1)
                    atom.SetIsotope(0); atom.SetAtomMapNum(0)
                
                try: clean_ion = Chem.RemoveHs(rw_ion)
                except: clean_ion = rw_ion.GetMol()
                
                metal_smi = utils.get_standardized_metal_smiles(clean_ion)
                if metal_smi:
                    type_tag = ""
                    if ion.HasProp("_Name"):
                        name_prop = ion.GetProp("_Name")
                        if name_prop.endswith("**"): type_tag = "**"
                        elif name_prop.endswith("***"): type_tag = "***"
                    if type_tag: smiles_list.append(metal_smi + type_tag)
                    else: smiles_list.append(metal_smi + "***")
                else:
                    smi = Chem.MolToSmiles(clean_ion, canonical=True)
                    if smi:
                        if ion.HasProp("_Name"):
                            if ion.GetProp("_Name").endswith("**"): smi += "**"
                            else: smi += "***"
                        else: smi += "***"
                        smiles_list.append(smi)
            except: pass

        if not main_mol or main_mol.GetNumAtoms() == 0: return smiles_list

        initial_shards = utils.execute_fragmentation_pipeline(main_mol, pre_calc_data['bond_info'])
        processing_queue = list(initial_shards)
        max_iterations, iterations = 1000, 0

        while processing_queue and iterations < max_iterations:
            current_frag = processing_queue.pop(0)
            iterations += 1
            
            rw_mol = Chem.RWMol(current_frag)
            dummies = [a for a in rw_mol.GetAtoms() if a.GetAtomicNum() == 0]
            
            for atom in dummies:
                for bond in atom.GetBonds():
                    if bond.GetBondType() != Chem.BondType.SINGLE:
                        bond.SetBondType(Chem.BondType.SINGLE)
                        bond.SetIsAromatic(False)
                atom.SetAtomicNum(1); atom.SetIsotope(0); atom.SetAtomMapNum(0)
                if atom.HasProp("_original_index"): atom.ClearProp("_original_index")
                if atom.HasProp("_cut_id"): atom.ClearProp("_cut_id")
                for neighbor in atom.GetNeighbors():
                    neighbor.SetNoImplicit(False); neighbor.SetNumExplicitHs(0); neighbor.UpdatePropertyCache(strict=False)
            
            clean_temp = rw_mol.GetMol()
            rescued_by_cutting = False
            
            try: Chem.SanitizeMol(clean_temp)
            except:
                try: Chem.SanitizeMol(clean_temp, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_PROPERTIES)
                except: 
                    clean_temp = utils.advanced_rescue_valence_errors(clean_temp)
                    rescued_by_cutting = True
                    try: Chem.SanitizeMol(clean_temp, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_PROPERTIES)
                    except: pass
                
            try: clean_fragment = Chem.RemoveHs(clean_temp)
            except: clean_fragment = clean_temp

            if rescued_by_cutting:
                split_frags = list(Chem.GetMolFrags(clean_fragment, asMols=True))
                if len(split_frags) > 1:
                    processing_queue[0:0] = split_frags
                    continue 
                
            if clean_fragment.GetNumHeavyAtoms() == 0: continue

            pipeline_frags = [clean_fragment]
            try:
                temp_dist_mat = Chem.rdmolops.GetDistanceMatrix(clean_fragment)
                temp_eccs = temp_dist_mat.max(axis=1)
                temp_bond_info = utils.get_ring_bond_info(clean_fragment, temp_eccs)
            except: temp_bond_info = {}
            
            try: r3_results = utils.rule3_cut_large_rings(pipeline_frags, temp_bond_info)
            except: r3_results = pipeline_frags
            pipeline_frags = r3_results
            
            try: r5_results = utils.rule5_cut_acyclic_chains(pipeline_frags)
            except: r5_results = pipeline_frags
            
            has_new_dummy = False
            for res_mol in r5_results:
                for atom in res_mol.GetAtoms():
                    if atom.GetAtomicNum() == 0:
                        has_new_dummy = True
                        break
                if has_new_dummy: break
            
            if has_new_dummy:
                processing_queue[0:0] = r5_results
            else:
                final_processed_mol = r5_results[0]
                if not utils.is_fragment_valid(final_processed_mol): continue 

                metal_smi = utils.get_standardized_metal_smiles(final_processed_mol)
                if metal_smi: smiles_list.append(metal_smi)
                else:
                    try:
                        final_smi = Chem.MolToSmiles(final_processed_mol, canonical=True)
                        if final_smi: smiles_list.append(final_smi)
                    except: pass
    except Exception: pass
    return smiles_list

# --- 4. �� 多进程 Worker (处理单个复合物) ---
def worker_process_complex(task):
    """此函数在独立的 CPU 核心上运行，完成单复合物的加载与切割"""
    import utils
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog('rdApp.*')
    
    result = {'success': False, 'smiles': []}
    smiles_set = set()
    
    try:
        # A. 兼容加载 Ligand (支持 PDB, SDF, MOL2)
        ligand_mol = None
        if 'ligand_sdf' in task and os.path.exists(task['ligand_sdf']):
            try: ligand_mol = Chem.MolFromMolFile(task['ligand_sdf'], removeHs=True, sanitize=False)
            except: pass
        if (ligand_mol is None or ligand_mol.GetNumAtoms() == 0) and 'ligand_mol2' in task and os.path.exists(task['ligand_mol2']):
            try: ligand_mol = Chem.MolFromMol2File(task['ligand_mol2'], removeHs=True, sanitize=False)
            except: pass
        if (ligand_mol is None or ligand_mol.GetNumAtoms() == 0) and 'ligand_path' in task and os.path.exists(task['ligand_path']):
            try: ligand_mol = Chem.MolFromPDBFile(task['ligand_path'], removeHs=True, sanitize=False)
            except: pass
            
        if not ligand_mol or ligand_mol.GetNumHeavyAtoms() > MAX_HEAVY_ATOMS_FOR_SMALL_MOL: 
            return result
            
        ligand_mol = utils.advanced_rescue_valence_errors(ligand_mol)
        try:
            ligand_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(ligand_mol, sanitizeOps=Chem.SANITIZE_SYMMRINGS|Chem.SANITIZE_SETCONJUGATION|Chem.SANITIZE_SETHYBRIDIZATION)
        except: return result

        # B. 兼容加载 Pocket (支持 BioLiP 和 PDBbind 的路径规则)
        pocket_mol = None
        if 'pocket_pdb' in task and os.path.exists(task['pocket_pdb']):
            try: pocket_mol = Chem.MolFromPDBFile(task['pocket_pdb'], removeHs=True, sanitize=False)
            except: pass
        elif 'pocket_path' in task and os.path.exists(task['pocket_path']):
            try: pocket_mol = Chem.MolFromPDBFile(task['pocket_path'], removeHs=True, sanitize=False)
            except: pass
            
        if not pocket_mol or pocket_mol.GetNumAtoms() == 0: return result

        pocket_mol = utils.advanced_rescue_valence_errors(pocket_mol)
        raw_pocket_frags = Chem.GetMolFrags(pocket_mol, asMols=True, sanitizeFrags=False)
        pocket_frags_clean = []
        for f in raw_pocket_frags:
            is_water = False
            if f.GetNumAtoms() == 1:
                atom = f.GetAtomWithIdx(0)
                if atom.GetAtomicNum() == 8 and atom.GetFormalCharge() == 0: is_water = True
            if not is_water: pocket_frags_clean.append(f)
            
        if not pocket_frags_clean: return result

        # C. 拓扑切割
        all_mols = list(Chem.GetMolFrags(ligand_mol, asMols=True, sanitizeFrags=False)) + pocket_frags_clean
        complex_has_valid = False
        
        for mol in all_mols:
            shards_smiles = fast_get_shards_smiles(mol)
            if shards_smiles:
                complex_has_valid = True
                for smi in shards_smiles:
                    if smi: smiles_set.add(smi)
        
        if complex_has_valid:
            result['success'] = True
            result['smiles'] = list(smiles_set)
            
    except Exception: pass
    return result

# --- 5. PDBbind 专属解析工具 ---
def parse_pdbbind_index(file_path, complex_type):
    records = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#'): continue
                parts = re.findall(r'(\S+)', line)
                if len(parts) >= 5: records.append({'pdb_id': parts[0], 'release_year': int(parts[2]), 'type': complex_type})
    except: pass
    return records

def get_complex_path(row, pl_base_dir, nl_base_dir):
    if row['type'] == 'NL': return os.path.join(nl_base_dir, row['pdb_id'])
    elif row['type'] == 'PL':
        year = row['release_year']
        if 1981 <= year <= 2000: year_folder = '1981-2000'
        elif 2001 <= year <= 2010: year_folder = '2001-2010'
        else: year_folder = '2011-2019'
        return os.path.join(pl_base_dir, year_folder, row['pdb_id'])
    return None

print(f"✅ 单元格 1 启动完毕：模块加载成功！系统核心数: {DETECTED_CORES}，已分配切割并发线程: {NUM_WORKERS}")

✅ 单元格 1 启动完毕：模块加载成功！系统核心数: 32，已分配切割并发线程: 32


In [ ]:
# =============================================================================
# 单元格 2: BioLiP 并行词汇表生成 (Pebble 抗毁版 + 768维嵌入)
# =============================================================================

NUM_BIOLIP_TO_PROCESS = None  # 测试可设为 1000
BIOLIP_DATA_ROOT = os.path.join('datas', 'tain_datas', 'SE3TD', 'BioLiP', 'datas_BioLiP')

print(f"[{'='*50}]\n>>> 开始扫描 BioLiP 数据目录: {BIOLIP_DATA_ROOT}\n[{'='*50}]")
if not os.path.exists(BIOLIP_DATA_ROOT):
    raise FileNotFoundError(f"找不到数据目录，请检查路径。")

all_complex_dirs = [d for d in os.listdir(BIOLIP_DATA_ROOT) if os.path.isdir(os.path.join(BIOLIP_DATA_ROOT, d))]
task_list = []

for complex_id in tqdm(all_complex_dirs, desc="构建任务列表"):
    complex_dir = os.path.join(BIOLIP_DATA_ROOT, complex_id)
    receptor_files = glob.glob(os.path.join(complex_dir, "*_receptor.pdb"))
    if not receptor_files: continue 
    
    ligand_files = glob.glob(os.path.join(complex_dir, "*_ligand.pdb"))
    for lig_path in ligand_files:
        task_list.append({'pdb_id': complex_id, 'ligand_path': lig_path, 'pocket_path': receptor_files[0]})

if NUM_BIOLIP_TO_PROCESS: task_list = task_list[:NUM_BIOLIP_TO_PROCESS]

print(f"\n成功匹配 {len(task_list)} 个复合物。启动 Pebble 多进程并发切割 (Workers={NUM_WORKERS})...")
unique_smiles = set()
success_count = 0

# �� 使用 Pebble 进程池进行物理级隔离与超时强杀
with ProcessPool(max_workers=NUM_WORKERS) as pool:
    # 设定 60 秒物理超时，防止 C++ 死锁
    future = pool.map(worker_process_complex, task_list, timeout=60)
    iterator = future.result()
    
    pbar = tqdm(total=len(task_list), desc="并行拓扑切割")
    
    while True:
        try:
            res = next(iterator)
            if res['success']:
                success_count += 1
                unique_smiles.update(res['smiles'])
        except StopIteration:
            break  # 遍历完成
        except TimeoutError:
            pass  # 忽略超时死锁分子
        except Exception:
            pass  # 忽略任何 C++ 导致的子进程暴毙
        finally:
            pbar.update(1) # 无论成功失败，进度条必须推进
            
    pbar.close()

print(f"\n>>> 拓扑切割完成! 有效复合物: {success_count}/{len(task_list)}")
print(f">>> 提取到独立碎片 (SMILES) 共计: {len(unique_smiles)} 个")

print(f"\n启动 ChemBERTa 模型提取 768 维特征...")
if not os.path.exists(LOCAL_MODEL_PATH):
    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
    model = AutoModel.from_pretrained(HF_MODEL_NAME)
    os.makedirs(LOCAL_MODEL_PATH, exist_ok=True)
    tokenizer.save_pretrained(LOCAL_MODEL_PATH)
    model.save_pretrained(LOCAL_MODEL_PATH)
else:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
    model = AutoModel.from_pretrained(LOCAL_MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

sorted_smiles = sorted(list(unique_smiles))
vocab_dict = {"<UNK>": [0.0] * TARGET_EMBEDDING_DIM}

def get_768_embedding(smi):
    try:
        inputs = tokenizer(smi, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad(): outputs = model(**inputs)
        return outputs.last_hidden_state[:, 0, :].squeeze().cpu().tolist()
    except: return [0.0] * TARGET_EMBEDDING_DIM

for smi in tqdm(sorted_smiles, desc="ChemBERTa 推理中"):
    vocab_dict[smi] = get_768_embedding(smi)

print(f"\n正在将片段及其 768 维特征写入 JSON...")
try:
    with open(OUTPUT_VOCAB_768, 'w', encoding='utf-8') as f:
        f.write("{\n")
        unk_vec = vocab_dict.pop("<UNK>")
        f.write(f'  "<UNK>": {json.dumps(unk_vec)}')
        sorted_items = sorted(vocab_dict.items()) 
        if sorted_items:
            f.write(",\n")
            for i, (smi, vec) in enumerate(sorted_items):
                line = f'  {json.dumps(smi, ensure_ascii=False)}: {json.dumps(vec)}'
                if i < len(sorted_items) - 1: f.write(f"{line},\n")
                else: f.write(f"{line}\n")
        else: f.write("\n")
        f.write("}")
    print(f"�� BioLiP 词汇表生成结束! 已保存至:\n => {os.path.abspath(OUTPUT_VOCAB_768)}")
except Exception as e:
    print(f"写入文件失败: {e}")

In [3]:
# =============================================================================
# 单元格 3: PDBbind 并行补充词汇表 (Pebble 抗毁版 + 增量提取)
# =============================================================================

NUM_PDBBIND_TO_PROCESS = None  # 测试可设为 1000
PDBBIND_ROOT = os.path.join('datas', 'tain_datas', 'SE3TD', 'PDBbind')
pl_index_path = os.path.join(PDBBIND_ROOT, 'index', 'INDEX_general_PL.2020R1.lst')
nl_index_path = os.path.join(PDBBIND_ROOT, 'index', 'INDEX_general_NL.2020R1.lst')
pl_data_dir = os.path.join(PDBBIND_ROOT, 'P-L')
nl_data_dir = os.path.join(PDBBIND_ROOT, 'NA-L')

print(f"[{'='*50}]\n>>> 开始扫描 PDBbind 数据集以增量扩充词汇表\n[{'='*50}]")

if os.path.exists(OUTPUT_VOCAB_768):
    print(f"1. 成功读取已有词汇表: {OUTPUT_VOCAB_768}")
    with open(OUTPUT_VOCAB_768, 'r', encoding='utf-8') as f:
        vocab_dict = json.load(f)
else:
    print(f"警告: 未找到 {OUTPUT_VOCAB_768}，将创建新字典。")
    vocab_dict = {"<UNK>": [0.0] * 768}

existing_smiles_set = set(vocab_dict.keys())
print(f"   => 当前词汇表已包含 {len(existing_smiles_set)} 个片段。")

pl_data = parse_pdbbind_index(pl_index_path, 'PL')
nl_data = parse_pdbbind_index(nl_index_path, 'NL')
df_full = pd.DataFrame(pl_data + nl_data)

df_full['complex_path'] = df_full.apply(lambda r: get_complex_path(r, pl_data_dir, nl_data_dir), axis=1)
df_full['ligand_sdf'] = df_full.apply(lambda r: os.path.join(r['complex_path'], f"{r['pdb_id']}_ligand.sdf"), axis=1)
df_full['ligand_mol2'] = df_full.apply(lambda r: os.path.join(r['complex_path'], f"{r['pdb_id']}_ligand.mol2"), axis=1)
df_full['pocket_pdb'] = df_full.apply(lambda r: os.path.join(r['complex_path'], f"{r['pdb_id']}_pocket.pdb"), axis=1)

if NUM_PDBBIND_TO_PROCESS: df_to_process = df_full.head(NUM_PDBBIND_TO_PROCESS).copy()
else: df_to_process = df_full.copy()
    
task_list = df_to_process.to_dict('records')
print(f"\n2. 准备并行处理 {len(task_list)} 个 PDBbind 复合物...")

unique_new_smiles = set()
success_count = 0

# �� 使用 Pebble 进程池进行物理级隔离与超时强杀
with ProcessPool(max_workers=NUM_WORKERS) as pool:
    future = pool.map(worker_process_complex, task_list, timeout=60)
    iterator = future.result()
    
    pbar = tqdm(total=len(task_list), desc="并行拓扑切割")
    
    while True:
        try:
            res = next(iterator)
            if res['success']:
                success_count += 1
                for smi in res['smiles']:
                    if smi not in existing_smiles_set:
                        unique_new_smiles.add(smi)
        except StopIteration:
            break
        except TimeoutError:
            pass
        except Exception:
            pass
        finally:
            pbar.update(1)
            
    pbar.close()

print(f"\n3. 切割完成! 有效复合物: {success_count}/{len(task_list)}")
print(f"   => 发现特有新片段: {len(unique_new_smiles)} 个")

if unique_new_smiles:
    print(f"\n4. 启动 ChemBERTa 进行增量提取...")
    if 'model' not in globals() or 'tokenizer' not in globals():
        tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
        model = AutoModel.from_pretrained(LOCAL_MODEL_PATH)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval()
    
    def get_768_embedding_inc(smi):
        try:
            inputs = tokenizer(smi, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            with torch.no_grad(): outputs = model(**inputs)
            return outputs.last_hidden_state[:, 0, :].squeeze().cpu().tolist()
        except: return [0.0] * 768

    sorted_new_smiles = sorted(list(unique_new_smiles))
    for smi in tqdm(sorted_new_smiles, desc="增量推理中"):
        vocab_dict[smi] = get_768_embedding_inc(smi)

    print(f"\n5. 正在覆盖写入 JSON...")
    try:
        with open(OUTPUT_VOCAB_768, 'w', encoding='utf-8') as f:
            f.write("{\n")
            unk_vec = vocab_dict.pop("<UNK>")
            f.write(f'  "<UNK>": {json.dumps(unk_vec)}')
            sorted_items = sorted(vocab_dict.items()) 
            if sorted_items:
                f.write(",\n")
                for i, (smi, vec) in enumerate(sorted_items):
                    line = f'  {json.dumps(smi, ensure_ascii=False)}: {json.dumps(vec)}'
                    if i < len(sorted_items) - 1: f.write(f"{line},\n")
                    else: f.write(f"{line}\n")
            else: f.write("\n")
            f.write("}")
        print(f"�� 增量更新结束! 词汇表已保存至:\n => {os.path.abspath(OUTPUT_VOCAB_768)}")
    except Exception as e:
        print(f"写入文件失败: {e}")
else:
    print("\n�� PDBbind 中没有产生新的片段，词汇表无需更新！")

[==================================================]
>>> 开始扫描 PDBbind 数据集以增量扩充词汇表
[==================================================]
警告: 未找到 processed_data/fragment_vocabulary_768.json，将创建新字典。
   => 当前词汇表已包含 1 个片段。


ValueError: Cannot set a DataFrame without columns to the column complex_path

In [ ]:
# =============================================================================
# 单元格 4: PCA 数据降维 (工业级：支持模型持久化与增量转换)
# =============================================================================

import os
import json
import numpy as np
import joblib  # 专门用于保存 sklearn 模型的库
from sklearn.decomposition import PCA

TARGET_PCA_DIM = 128
OUTPUT_VOCAB_768 = os.path.join('processed_data', 'fragment_vocabulary_768.json')
OUTPUT_VOCAB_PCA = os.path.join('processed_data', 'fragment_vocabulary_128_pca.json')

# [新增] PCA 模型文件的保存路径
PCA_MODEL_PATH = os.path.join('processed_data', 'pca_model_128.pkl')

print(f"[{'='*50}]\n>>> 开始执行 PCA 数据降维与映射\n[{'='*50}]")

with open(OUTPUT_VOCAB_768, 'r', encoding='utf-8') as f:
    vocab_768 = json.load(f)

smiles_list = []
vectors_list = []
for smi, vec in vocab_768.items():
    if smi != "<UNK>":
        smiles_list.append(smi)
        vectors_list.append(vec)

X = np.array(vectors_list)
print(f"   => 加载了 {X.shape[0]} 个 768 维特征。")

# =========================================================
# 核心逻辑：判断是“首次拟合”还是“加载旧模型做映射”
# =========================================================
if not os.path.exists(PCA_MODEL_PATH):
    print(f"\n2. [首次运行] 正在拟合全局 PCA 模型 (目标维度: {TARGET_PCA_DIM}) ...")
    pca = PCA(n_components=TARGET_PCA_DIM, random_state=42) 
    
    # 首次必须用 fit_transform
    X_pca = pca.fit_transform(X)
    
    # 保存这个拟合好的 PCA 矩阵 (非常关键！)
    joblib.dump(pca, PCA_MODEL_PATH)
    print(f"   => PCA 模型已永久冻结并保存至: {PCA_MODEL_PATH}")
    
    explained_variance = np.sum(pca.explained_variance_ratio_) * 100
    print(f"   => 保留了原空间 {explained_variance:.2f}% 的方差。")
else:
    print(f"\n2. [增量模式] 发现已冻结的 PCA 模型，正在加载以保证特征空间对齐...")
    # 加载之前固定的坐标系
    pca = joblib.load(PCA_MODEL_PATH)
    
    # 绝对不能再 fit 了！只能 transform！这样老词汇的向量绝对不会变
    X_pca = pca.transform(X)
    print(f"   => 已使用固定的投影矩阵完成 128 维映射。")

print(f"\n3. 组装并保存最终的 PCA 词汇表...")
vocab_128_pca = {"<UNK>": [0.0] * TARGET_PCA_DIM}

for i, smi in enumerate(smiles_list):
    vocab_128_pca[smi] = X_pca[i].tolist()

with open(OUTPUT_VOCAB_PCA, 'w', encoding='utf-8') as f:
    f.write("{\n")
    unk_vec = vocab_128_pca.pop("<UNK>")
    f.write(f'  "<UNK>": {json.dumps(unk_vec)}')
    sorted_items = sorted(vocab_128_pca.items()) 
    if sorted_items:
        f.write(",\n")
        for i, (smi, vec) in enumerate(sorted_items):
            line = f'  {json.dumps(smi, ensure_ascii=False)}: {json.dumps(vec)}'
            if i < len(sorted_items) - 1: f.write(f"{line},\n")
            else: f.write(f"{line}\n")
    else: f.write("\n")
    f.write("}")
    
print(f"�� 降维工作就绪! 128 维特征库已更新:\n => {os.path.abspath(OUTPUT_VOCAB_PCA)}")

In [ ]:
import os
import json
import numpy as np

# 指向刚刚生成的 128维 PCA 词汇表文件
OUTPUT_VOCAB_PCA = os.path.join('processed_data', 'fragment_vocabulary_128_pca.json')

if not os.path.exists(OUTPUT_VOCAB_PCA):
    print(f"❌ 找不到文件: {OUTPUT_VOCAB_PCA}\n请确保你已经成功运行了 单元格 4。")
else:
    print(f"正在从硬盘加载词汇表: {OUTPUT_VOCAB_PCA}")
    with open(OUTPUT_VOCAB_PCA, 'r', encoding='utf-8') as f:
        vocab_128_pca = json.load(f)

    # 提取所有的 128 维向量
    vectors = np.array(list(vocab_128_pca.values()))
    
    # 沿着 batch 维度去重
    unique_vectors = np.unique(vectors, axis=0)

    print(f"--------------------------------------------------")
    print(f"原始词汇表条数 (包含 <UNK>): {len(vectors)}")
    print(f"降维后唯一向量条数:          {len(unique_vectors)}")

    if len(vectors) == len(unique_vectors):
        print("✅ 验证完美通过：降维后没有任何两个片段产生完全相同的向量！")
    else:
        print(f"⚠️ 警告：检测到有 {len(vectors) - len(unique_vectors)} 组向量发生了碰撞！")
    print(f"--------------------------------------------------")